In [1]:
"""
EEG Channel Hierarchical Clustering
====================================
Input : inter_to_ict_chb01_03_2980_3010_adjacency_sparse.npz
        (176,640 x 176,640 sparse adjacency matrix)
        Structure: 23 EEG channels x 7,680 timepoints (30s @ 256 Hz)
        Node ordering: channel-major  ->  node i = channel (i//7680), timepoint (i%7680)
 
Output: dendrogram + correlation heatmap showing which EEG channels
        cluster together based on their temporal connectivity patterns.
"""

'\nEEG Channel Hierarchical Clustering\n====================================\nInput : inter_to_ict_chb01_03_2980_3010_adjacency_sparse.npz\n        (176,640 x 176,640 sparse adjacency matrix)\n        Structure: 23 EEG channels x 7,680 timepoints (30s @ 256 Hz)\n        Node ordering: channel-major  ->  node i = channel (i//7680), timepoint (i%7680)\n \nOutput: dendrogram + correlation heatmap showing which EEG channels\n        cluster together based on their temporal connectivity patterns.\n'

In [2]:
import numpy as np
import scipy.sparse as sp
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [3]:
CHANNEL_NAMES = [
    "FP1-F7", "F7-T7",  "T7-P7",  "P7-O1",
    "FP1-F3", "F3-C3",  "C3-P3",  "P3-O1",
    "FP2-F4", "F4-C4",  "C4-P4",  "P4-O2",
    "FP2-F8", "F8-T8",  "T8-P8",  "P8-O2",
    "FZ-CZ",  "CZ-PZ",
    "P7-T7",  "T7-FT9", "FT9-FT10","FT10-T8",
    "T8-P8-1"
]

In [6]:
N_CHANNELS  = 23
N_TIMEPOINTS = 7680   # 30 s × 256 Hz
 
# ── 1. Load the sparse adjacency matrix ────────────────────────────────────
print("Loading adjacency matrix …")
mat = sp.load_npz("/Users/antoniaspoerk/Desktop/Digital Neuroscience /Social Media Analytics/epilepsy_pediatrics_EEG/data/graphs/adjacency_sparse/inter_to_ict_chb01_03_2980_3010_adjacency_sparse.npz")
print(f"  Shape : {mat.shape}  |  Non-zeros : {mat.nnz:,}")
 
# ── 2. Extract temporal connectivity profile per channel ───────────────────
#
# Each channel occupies a contiguous block of N_TIMEPOINTS rows/cols.
# The intra-channel block (same channel, different timepoints) carries
# small float weights that encode temporal similarity within that channel.
# Summing edge weights per timepoint gives a 1-D "activity profile".
#
print("Building channel temporal profiles …")
ch_temporal = np.zeros((N_CHANNELS, N_TIMEPOINTS), dtype=np.float64)
for ch in range(N_CHANNELS):
    s, e = ch * N_TIMEPOINTS, (ch + 1) * N_TIMEPOINTS
    block = mat[s:e, s:e]                          # intra-channel block
    ch_temporal[ch] = np.array(block.sum(axis=1)).flatten()
 
# ── 3. Compute 23×23 Pearson correlation matrix ────────────────────────────
print("Computing channel correlation matrix …")
corr = np.corrcoef(ch_temporal)                    # values in [0, 1]
 
# Convert correlation to distance  (1 - r  keeps range in [0, 1])
dist_matrix = 1.0 - corr
np.fill_diagonal(dist_matrix, 0.0)
dist_condensed = squareform(dist_matrix, checks=False)
 
# ── 4. Hierarchical clustering (Ward linkage) ──────────────────────────────
print("Running hierarchical clustering (Ward linkage) …")
linkage = sch.linkage(dist_condensed, method="ward")
 
# ── 5. Plot ────────────────────────────────────────────────────────────────
print("Plotting …")
fig = plt.figure(figsize=(16, 9), facecolor="#0f1117")
fig.suptitle(
    "EEG Channel Hierarchical Clustering  ·  CHB-01  ·  2980–3010 s (pre-ictal)",
    color="white", fontsize=14, fontweight="bold", y=0.98
)
 
gs = gridspec.GridSpec(
    1, 2,
    width_ratios=[1, 1.4],
    left=0.04, right=0.97,
    top=0.91, bottom=0.08,
    wspace=0.08
)
 
ax_dendro  = fig.add_subplot(gs[0])
ax_heatmap = fig.add_subplot(gs[1])
 
ACCENT   = "#7c6af7"   # purple
WARM     = "#f7936a"   # orange
BG       = "#0f1117"
PANEL_BG = "#1a1d27"
TEXT     = "#e0e0e0"
 
# ── Dendrogram ──────────────────────────────────────────────────────────────
ax_dendro.set_facecolor(PANEL_BG)
for spine in ax_dendro.spines.values():
    spine.set_visible(False)
 
dend = sch.dendrogram(
    linkage,
    labels=CHANNEL_NAMES,
    orientation="left",
    ax=ax_dendro,
    color_threshold=0.55 * linkage[-1, 2],   # auto-colour major clusters
    above_threshold_color=TEXT,
    leaf_font_size=8.5,
)
 
ax_dendro.tick_params(axis="x", colors=TEXT, labelsize=7)
ax_dendro.tick_params(axis="y", colors=TEXT, labelsize=8.5)
ax_dendro.set_xlabel("Ward distance  (1 − r)", color=TEXT, fontsize=9)
ax_dendro.set_title("Dendrogram", color=TEXT, fontsize=11, pad=8)
ax_dendro.xaxis.label.set_color(TEXT)
 
# Re-colour dendrogram lines
cluster_colors = [ACCENT, WARM, "#5ecfb1", "#e05c97", "#f5d547"]
unique_colors  = list(dict.fromkeys(
    c for c in dend["color_list"] if c != TEXT
))
color_map = {uc: cluster_colors[i % len(cluster_colors)]
             for i, uc in enumerate(unique_colors)}
color_map[TEXT] = TEXT
 
for coll in ax_dendro.collections:
    pass   # lines are rendered as LineCollections
 
for line in ax_dendro.get_lines():
    c = line.get_color()
    line.set_color(color_map.get(c, c))
    line.set_linewidth(1.6)
 
# ── Reordered correlation heatmap ──────────────────────────────────────────
ax_heatmap.set_facecolor(PANEL_BG)
for spine in ax_heatmap.spines.values():
    spine.set_visible(False)
 
order = dend["leaves"]
corr_ordered  = corr[np.ix_(order, order)]
labels_ordered = [CHANNEL_NAMES[i] for i in order]
 
im = ax_heatmap.imshow(
    corr_ordered,
    cmap="RdYlBu_r",
    vmin=0, vmax=1,
    aspect="auto",
    interpolation="nearest"
)
 
ax_heatmap.set_xticks(range(N_CHANNELS))
ax_heatmap.set_yticks(range(N_CHANNELS))
ax_heatmap.set_xticklabels(labels_ordered, rotation=90, fontsize=7.5, color=TEXT)
ax_heatmap.set_yticklabels(labels_ordered, fontsize=7.5, color=TEXT)
ax_heatmap.set_title("Channel Correlation Matrix  (reordered by cluster)", color=TEXT, fontsize=11, pad=8)
 
cbar = fig.colorbar(im, ax=ax_heatmap, fraction=0.035, pad=0.02)
cbar.set_label("Pearson r", color=TEXT, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=TEXT, labelcolor=TEXT)
cbar.outline.set_edgecolor(PANEL_BG)
 
# ── Save ───────────────────────────────────────────────────────────────────
out_path = "eeg_clustering.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=BG)
print(f"Saved → {out_path}")
plt.close(fig)
print("Done.")

Loading adjacency matrix …
  Shape : (176640, 176640)  |  Non-zeros : 3,956,402
Building channel temporal profiles …
Computing channel correlation matrix …
Running hierarchical clustering (Ward linkage) …
Plotting …
Saved → eeg_clustering.png
Done.
